# T05 — Embedding Foundation Model AlphaEarth vs Fitur Spektral Sentinel-2: Efisiensi Label dan Transfer Antarwilayah di Kawasan PSN Merauke

| Tahap | Isi | Waktu |
|---|---|---|
| 0–1 | Instalasi, login Earth Engine, konfigurasi | 2 menit |
| 2 | Blok C dan D (peringkat 3 dan 4 kehilangan pohon 2024–2025), tahun embedding terbaru | 2 menit |
| 3 | Empat set fitur: S2-tahunan, S2-musiman, AlphaEarth (AE), AE + S2-musiman | 1 menit |
| 4 | Label konsensus 4–5 kelas tanpa survei lapangan, sampel berstrata kedua blok | 5–10 menit |
| 5 | **E1** efisiensi label (5–semua titik per kelas × 10 ulangan × 3 algoritma) | 10–20 menit |
| 6 | **E2** transfer antarblok (latih C → uji D, dan sebaliknya) + McNemar | 2 menit |
| 7 | **E3** klasterisasi tanpa label (k-means: ARI, NMI) + t-SNE | 3 menit |
| 8 | Peta klasifikasi AE vs S2 di Blok C (GEE) | 5 menit |
| 9 | Ringkasan dan unduh `T05_hasil.zip` | 1 menit |

Jalankan **Runtime → Jalankan semua**. Bila ada error, kirim pesan error-nya.

In [ ]:
# 0. Instalasi
!pip install -q earthengine-api scikit-learn pandas matplotlib seaborn scipy openpyxl requests

In [ ]:
# 1. Earth Engine
import ee
GEE_PROJECT = 'riset-psn-pps'
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate(force=True); ee.Initialize(project=GEE_PROJECT)
print('Earth Engine siap:', ee.String('OK').getInfo())

In [ ]:
# Konfigurasi
import os, json, time, zipfile, io, requests
import numpy as np, pandas as pd
TOPIC = 'T05'; SEED = 42
OUT = f'/content/{TOPIC}_hasil' if os.path.exists('/content') else f'./{TOPIC}_hasil'
DIR_DATA, DIR_TAB, DIR_FIG = [os.path.join(OUT, d) for d in ('07_dataset', '03_tabel', '02_gambar')]
for d in (DIR_DATA, DIR_TAB, DIR_FIG): os.makedirs(d, exist_ok=True)
CRS = 'EPSG:32754'; BLOCK_KM = 20
N_PER_CLASS = 500                     # sampel per kelas per blok
MIN_PIX = 2000                        # kelas dengan piksel kandidat < MIN_PIX (10 m) di salah satu blok tidak dipakai
SIZES = [5, 10, 25, 50, 100, 250]     # jumlah titik latih per kelas (E1); 'semua' ditambahkan otomatis
REPS = 10
CLASS_ID = {1: 'Hutan', 2: 'Savana / semak / rumput', 3: 'Lahan basah', 4: 'Lahan dibuka', 5: 'Pertanian'}
CLASS_EN = {1: 'Forest', 2: 'Savanna / shrub / grass', 3: 'Wetland', 4: 'Cleared land', 5: 'Cropland'}
FS_NAME = {'S2a': 'S2-tahunan', 'S2s': 'S2-musiman', 'AE': 'AlphaEarth', 'AE+S2s': 'AlphaEarth + S2-musiman'}
FS_EN = {'S2a': 'S2-annual', 'S2s': 'S2-seasonal', 'AE': 'AlphaEarth', 'AE+S2s': 'AlphaEarth + S2-seasonal'}
COLS_EN = {'set_fitur': 'feature_set', 'algoritma': 'algorithm', 'n_per_kelas': 'n_per_class', 'ulangan': 'repeat',
           'F1_makro': 'macro_F1', 'F1_makro_rata2': 'macro_F1_mean', 'F1_makro_sd': 'macro_F1_sd', 'OA_rata2': 'OA_mean',
           'arah': 'direction', 'latih': 'train', 'uji': 'test', 'F1_dalam_blok': 'F1_within_block', 'F1_transfer': 'F1_transfer',
           'penurunan': 'drop', 'pasangan': 'pair', 'kelas': 'class', 'blok': 'block', 'piksel_10m': 'pixels_10m', 'luas_ha': 'area_ha',
           'dipakai': 'used', 'n_latih': 'n_train', 'n_uji': 'n_test', 'p_nilai': 'p_value', 'signifikan': 'significant',
           'jumlah_fitur': 'n_features', 'isi': 'content', 'rasio_label': 'label_ratio', 'peringkat': 'rank', 'loss_2024_2025_ha': 'loss_2024_2025_ha',
           'presisi': 'precision', 'recall': 'recall', 'n': 'n', 'keterangan': 'note', 'nilai': 'value', 'metrik': 'metric', 'luas_peta_ha': 'mapped_area_ha'}
VAL_EN = {**{v: CLASS_EN[k] for k, v in CLASS_ID.items()}, **{v: FS_EN[k] for k, v in FS_NAME.items()}, 'semua': 'all', 'ya': 'yes', 'tidak': 'no'}
def save_tab(df, name_id, name_en):
    d = df.copy()
    for c in d.columns:
        if not pd.api.types.is_numeric_dtype(d[c]): d[c] = d[c].map(lambda v: VAL_EN.get(v, v))
    df.to_csv(os.path.join(DIR_TAB, f'T05_tabel_{name_id}_ID.csv'), index=False)
    d.rename(columns=COLS_EN).to_csv(os.path.join(DIR_TAB, f'T05_table_{name_en}_EN.csv'), index=False)
np.random.seed(SEED); print('Folder keluaran:', OUT)

## Tahap 2 — Blok C dan D, serta tahun embedding
Grid 20 × 20 km yang sama dengan T01/T02. Blok A dan B sudah dipakai T01 dan T02, sehingga T05 memakai **Blok C (peringkat 3)** sebagai wilayah utama dan **Blok D (peringkat 4)** sebagai wilayah uji transfer. Tahun embedding (`Y`) adalah tahun AlphaEarth terbaru yang tersedia di kedua blok.

In [ ]:
kab = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Merauke')); kab_geom = kab.geometry()
gfc = ee.Image('UMD/hansen/global_forest_change_2025_v1_13'); lossyear = gfc.select('lossyear').unmask(0); tc2000 = gfc.select('treecover2000')
loss_ha = ee.Image.pixelArea().divide(1e4).updateMask(lossyear.gte(24).And(lossyear.lte(25))).rename('loss_ha')
grid = kab_geom.coveringGrid(ee.Projection(CRS).atScale(BLOCK_KM * 1000))
grid_sorted = loss_ha.reduceRegions(collection=grid, reducer=ee.Reducer.sum(), scale=30, tileScale=8).filter(ee.Filter.gt('sum', 0)).sort('sum', False)
top = grid_sorted.limit(4).getInfo()['features']
blocks = pd.DataFrame([{'peringkat': i + 1, 'blok': 'ABCD'[i], 'loss_2024_2025_ha': round(f['properties']['sum'], 1)} for i, f in enumerate(top)])
print(blocks.to_string(index=False)); save_tab(blocks, 'kandidat_blok', 'candidate_blocks')
BLK = {'C': ee.Geometry(top[2]['geometry']), 'D': ee.Geometry(top[3]['geometry'])}
for b, g in BLK.items():
    c = g.centroid(1).coordinates().getInfo(); print(f'Blok {b}: pusat lon {c[0]:.4f}, lat {c[1]:.4f}')
    json.dump({'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'properties': {'blok': b}, 'geometry': g.transform('EPSG:4326', 1).getInfo()}]},
              open(os.path.join(DIR_DATA, f'T05_blok{b}.geojson'), 'w'))

AEC = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
def ae_year_ok(y):
    return all(AEC.filterDate(f'{y}-01-01', f'{y + 1}-01-01').filterBounds(g).size().getInfo() > 0 for g in BLK.values())
Y = 2025 if ae_year_ok(2025) else 2024
print('Tahun embedding AlphaEarth yang dipakai: Y =', Y)

## Tahap 3 — Empat set fitur
| Set | Isi | Jumlah |
|---|---|---|
| S2-tahunan | Median Sentinel-2 tahun Y: 10 band + 6 indeks | 16 |
| S2-musiman | Median musim hujan (Jan–Apr Y) dan kemarau (Jun–Okt Y): 10 band + 6 indeks masing-masing | 32 |
| AlphaEarth | Embedding tahunan Y, A00–A63 (vektor satuan) | 64 |
| AE + S2-musiman | Gabungan | 96 |

In [ ]:
BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']; IDX = ['NDVI', 'EVI', 'NDWI', 'NDMI', 'NBR', 'BSI']
cs_plus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
REG = BLK['C'].union(BLK['D'], 1).buffer(500)
def s2_med(a, b):
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(REG).filterDate(a, b).linkCollection(cs_plus, ['cs_cdf'])
           .map(lambda i: i.updateMask(i.select('cs_cdf').gte(0.6)).select(BANDS).divide(10000)))
    return col.median()
def with_idx(img):
    b = lambda n: img.select(n)
    return ee.Image.cat([img.select(BANDS), img.normalizedDifference(['B8', 'B4']).rename('NDVI'),
        img.expression('2.5*((N-R)/(N+6*R-7.5*B+1))', {'N': b('B8'), 'R': b('B4'), 'B': b('B2')}).rename('EVI'),
        img.normalizedDifference(['B3', 'B8']).rename('NDWI'), img.normalizedDifference(['B8', 'B11']).rename('NDMI'),
        img.normalizedDifference(['B8', 'B12']).rename('NBR'),
        img.expression('((S+R)-(N+B))/((S+R)+(N+B))', {'S': b('B11'), 'R': b('B4'), 'N': b('B8'), 'B': b('B2')}).rename('BSI')])
S2N = BANDS + IDX
s2a = with_idx(s2_med(f'{Y}-01-01', f'{Y + 1}-01-01')).rename([f'{x}_thn' for x in S2N])
s2w = with_idx(s2_med(f'{Y}-01-01', f'{Y}-05-01')).rename([f'{x}_hjn' for x in S2N])
s2d = with_idx(s2_med(f'{Y}-06-01', f'{Y}-11-01')).rename([f'{x}_kmr' for x in S2N])
AEB = [f'A{i:02d}' for i in range(64)]
ae = AEC.filterDate(f'{Y}-01-01', f'{Y + 1}-01-01').filterBounds(REG).mosaic().select(AEB)
FEAT = ee.Image.cat([s2a, s2w, s2d, ae]).toFloat()
FS = {'S2a': [f'{x}_thn' for x in S2N], 'S2s': [f'{x}_hjn' for x in S2N] + [f'{x}_kmr' for x in S2N], 'AE': AEB}
FS['AE+S2s'] = FS['AE'] + FS['S2s']
fs_tab = pd.DataFrame([{'set_fitur': FS_NAME[k], 'jumlah_fitur': len(v)} for k, v in FS.items()]); print(fs_tab.to_string(index=False))
save_tab(fs_tab, 'set_fitur', 'feature_sets')

## Tahap 4 — Label konsensus dan sampel
Label diambil dari produk independen: Hansen GFC, RADD, JRC TMF, ESA WorldCover 2021, dan JRC Global Surface Water. Hanya piksel yang **disepakati** beberapa produk yang diberi label.

| Kelas | Aturan |
|---|---|
| 1 Hutan | tutupan pohon 2000 ≥ 50%, tanpa kehilangan s.d. 2025, tanpa RADD, WorldCover = pohon |
| 2 Savana / semak / rumput | tutupan pohon 2000 < 30%, tanpa kehilangan, WorldCover = semak atau rumput, air < 10% |
| 3 Lahan basah | WorldCover = lahan basah herba atau mangrove, tanpa kehilangan, air < 50% |
| 4 Lahan dibuka | Hansen kehilangan tahun Y−1 s.d. Y (bila Y = 2025) atau Y−2 s.d. Y (bila Y = 2024), dikonfirmasi RADD atau TMF |
| 5 Pertanian | WorldCover = pertanian, tanpa kehilangan, air < 50% |

Kelas yang terlalu sedikit pikselnya di salah satu blok otomatis tidak dipakai.

In [ ]:
radd = (ee.ImageCollection('projects/radar-wur/raddalert/v1').filterMetadata('layer', 'contains', 'alert')
        .filterMetadata('geography', 'equals', 'asia').sort('system:time_end', False).first())
rdate = radd.select('Date').unmask(0)
tmf = ee.ImageCollection('projects/JRC/TMF/v1_2025/AnnualChanges').mosaic()
wc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
gsw = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').unmask(0)
yy = Y - 2000
LOSS_YEARS = [yy - 1, yy] if Y == 2025 else [yy - 2, yy - 1, yy]
loss_c = lossyear.gte(min(LOSS_YEARS)).And(lossyear.lte(max(LOSS_YEARS)))
radd_c = rdate.gte(min(LOSS_YEARS) * 1000 + 1).And(rdate.lte(max(LOSS_YEARS) * 1000 + 366))
tmf_c = tmf.select(f'Dec{2000 + max(LOSS_YEARS)}').remap([2, 3], [1, 1], 0)
no_loss = lossyear.eq(0); radd_any = rdate.gt(0)
c1 = tc2000.gte(50).And(no_loss).And(radd_any.Not()).And(wc.eq(10))
c2 = tc2000.lt(30).And(no_loss).And(wc.eq(20).Or(wc.eq(30))).And(gsw.lt(10))
c3 = wc.eq(90).Or(wc.eq(95)).And(no_loss).And(gsw.lt(50))
c4 = loss_c.And(radd_c.Or(tmf_c)).And(gsw.lt(50))
c5 = wc.eq(40).And(no_loss).And(gsw.lt(50))
label = ee.Image(0).where(c1, 1).where(c2, 2).where(c3, 3).where(c5, 5).where(c4, 4).selfMask().rename('label').toInt()

av = []
for b, g in BLK.items():
    h = label.reduceRegion(ee.Reducer.frequencyHistogram(), g, 10, maxPixels=1e10, tileScale=8).get('label').getInfo() or {}
    for k in CLASS_ID: av.append({'blok': b, 'kelas': CLASS_ID[k], 'label': k, 'piksel_10m': int(h.get(str(k), 0)), 'luas_ha': round(h.get(str(k), 0) * 0.01, 1)})
av = pd.DataFrame(av)
USE = [k for k in CLASS_ID if av[av.label == k].piksel_10m.min() >= MIN_PIX]
av['dipakai'] = av.label.map(lambda k: 'ya' if k in USE else 'tidak')
print(av.drop(columns='label').to_string(index=False)); print('Kelas dipakai:', [CLASS_ID[k] for k in USE])
save_tab(av.drop(columns='label'), 'ketersediaan_label', 'label_availability')

def fc_to_df(fc, page=500):
    n = fc.size().getInfo(); lst = fc.toList(n); rows = []
    for s in range(0, n, page):
        for f in ee.FeatureCollection(lst.slice(s, s + page)).getInfo()['features']:
            p = dict(f['properties']); p['lon'], p['lat'] = f['geometry']['coordinates']; rows.append(p)
    return pd.DataFrame(rows)
F_SAMPLE = os.path.join(DIR_DATA, 'T05_sampel.csv')
if os.path.exists(F_SAMPLE):
    df = pd.read_csv(F_SAMPLE); print('Sampel dimuat ulang:', len(df))
else:
    parts = []
    for b, g in BLK.items():
        t0 = time.perf_counter()
        fc = label.addBands(FEAT).stratifiedSample(numPoints=0, classBand='label', classValues=USE, classPoints=[N_PER_CLASS] * len(USE),
                                                   region=g, scale=10, seed=SEED, geometries=True, tileScale=8, dropNulls=True)
        d = fc_to_df(fc); d['blok'] = b; parts.append(d); print(f'Blok {b}: {len(d)} sampel ({time.perf_counter() - t0:.0f} dtk)')
    df = pd.concat(parts, ignore_index=True).dropna(subset=sum(FS.values(), []))
    df['label'] = df['label'].astype(int)
    df['grup'] = df.blok + '_' + np.floor(df.lon / 0.018).astype(int).astype(str) + '_' + np.floor(df.lat / 0.018).astype(int).astype(str)
    df.to_csv(F_SAMPLE, index=False)
print(pd.crosstab(df['label'].map(CLASS_ID), df.blok))

## Tahap 5 — E1: Efisiensi label di Blok C
* Data Blok C dibagi 70% kumpulan latih / 30% uji dengan **GroupShuffleSplit** berdasarkan sel 2 km. Data uji **tetap** untuk semua percobaan.
* Dari kumpulan latih diambil acak *n* titik per kelas (5, 10, 25, 50, 100, 250, semua), **10 ulangan** per *n*.
* Tiga algoritma dengan parameter tetap (tanpa tuning, agar adil untuk data kecil): **kNN** (k = 3, jarak kosinus untuk AE), **regresi logistik** (*linear probe*), **Random Forest** (200 pohon).

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support
def make_clf(name, fs):
    if name == 'kNN':
        return KNeighborsClassifier(n_neighbors=3, metric='cosine') if fs == 'AE' else Pipeline([('sc', StandardScaler()), ('m', KNeighborsClassifier(n_neighbors=3))])
    if name == 'LogReg':
        return Pipeline([('sc', StandardScaler()), ('m', LogisticRegression(max_iter=3000, C=1.0))])
    return RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
ALGS = ['kNN', 'LogReg', 'RF']
C = df[df.blok == 'C'].reset_index(drop=True); Dd = df[df.blok == 'D'].reset_index(drop=True)
tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED).split(C, C.label, C.grup))
POOL, TEST = C.iloc[tr_i].reset_index(drop=True), C.iloc[te_i].reset_index(drop=True)
print('Kumpulan latih:', POOL.label.value_counts().sort_index().to_dict(), '| uji:', TEST.label.value_counts().sort_index().to_dict())
def draw(pool, n, rng):
    return pd.concat([g.sample(min(n, len(g)), random_state=int(rng.integers(1e9))) if n != 'semua' else g for _, g in pool.groupby('label')])
rows = []; t0 = time.perf_counter(); rng = np.random.default_rng(SEED)
for n in SIZES + ['semua']:
    for r in range(REPS if n != 'semua' else 1):
        tr = draw(POOL, n, rng)
        for fs, cols in FS.items():
            for a in ALGS:
                m = make_clf(a, fs).fit(tr[cols].values, tr.label.values); yp = m.predict(TEST[cols].values)
                rows.append({'set_fitur': FS_NAME[fs], 'algoritma': a, 'n_per_kelas': n, 'ulangan': r + 1,
                             'F1_makro': f1_score(TEST.label, yp, average='macro'), 'OA': accuracy_score(TEST.label, yp)})
    print(f'n = {n}: selesai ({time.perf_counter() - t0:.0f} dtk)')
E1 = pd.DataFrame(rows); save_tab(E1.round(4), 'E1_efisiensi_label_mentah', 'E1_label_efficiency_raw')
E1s = (E1.groupby(['set_fitur', 'algoritma', 'n_per_kelas'], sort=False)
       .agg(F1_makro_rata2=('F1_makro', 'mean'), F1_makro_sd=('F1_makro', 'std'), OA_rata2=('OA', 'mean')).reset_index().round(4))
save_tab(E1s, 'E1_efisiensi_label_ringkas', 'E1_label_efficiency_summary')
print(E1s.pivot_table(index=['algoritma', 'set_fitur'], columns='n_per_kelas', values='F1_makro_rata2', sort=False).round(3).to_string())
# Jumlah label yang dibutuhkan S2 untuk menyamai AE pada n = 10 (rasio efisiensi label)
eff = []
for a in ALGS:
    ae10 = E1s[(E1s.set_fitur == FS_NAME['AE']) & (E1s.algoritma == a) & (E1s.n_per_kelas == 10)].F1_makro_rata2.iloc[0]
    for fs in ['S2a', 'S2s']:
        s = E1s[(E1s.set_fitur == FS_NAME[fs]) & (E1s.algoritma == a) & (E1s.n_per_kelas != 'semua')]
        hit = s[s.F1_makro_rata2 >= ae10]
        eff.append({'algoritma': a, 'set_fitur': FS_NAME[fs], 'F1_AE_n10': ae10, 'n_per_kelas': int(hit.n_per_kelas.iloc[0]) if len(hit) else '> 250',
                    'rasio_label': round(int(hit.n_per_kelas.iloc[0]) / 10, 1) if len(hit) else '> 25'})
eff = pd.DataFrame(eff); save_tab(eff, 'E1_label_setara_AE10', 'E1_labels_to_match_AE10'); print(eff.to_string(index=False))

## Tahap 6 — E2: Transfer antarblok dan uji McNemar
Model dilatih dengan **seluruh** kumpulan latih satu blok lalu diuji pada **blok lain** yang tidak pernah dilihat. Pembanding dalam-blok memakai data uji blok yang sama (pembagian spasial 70/30).

In [ ]:
from scipy.stats import binomtest
tr_d, te_d = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED).split(Dd, Dd.label, Dd.grup))
POOL_D, TEST_D = Dd.iloc[tr_d].reset_index(drop=True), Dd.iloc[te_d].reset_index(drop=True)
rows, PRED = [], {}
for (src, trn, tst_in, tst_out, dst) in [('C', POOL, TEST, Dd, 'D'), ('D', POOL_D, TEST_D, C, 'C')]:
    for fs, cols in FS.items():
        for a in ALGS:
            m = make_clf(a, fs).fit(trn[cols].values, trn.label.values)
            f_in = f1_score(tst_in.label, m.predict(tst_in[cols].values), average='macro')
            yp = m.predict(tst_out[cols].values); f_out = f1_score(tst_out.label, yp, average='macro'); PRED[(src, fs, a)] = yp
            rows.append({'arah': f'{src} → {dst}', 'set_fitur': FS_NAME[fs], 'algoritma': a, 'F1_dalam_blok': round(f_in, 4),
                         'F1_transfer': round(f_out, 4), 'penurunan': round(f_in - f_out, 4)})
E2 = pd.DataFrame(rows); save_tab(E2, 'E2_transfer_antarblok', 'E2_cross_block_transfer')
print(E2.pivot_table(index=['arah', 'algoritma'], columns='set_fitur', values='F1_transfer', sort=False).round(3).to_string())
def mcnemar(y, a, b):
    ca, cb = (a == y), (b == y); n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
    return n01, n10, (binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 > 0 else 1.0)
mc = []
for src, tst in (('C', Dd), ('D', C)):
    for a in ALGS:
        for f1_, f2_ in (('AE', 'S2s'), ('AE', 'S2a'), ('AE+S2s', 'AE')):
            n01, n10, p = mcnemar(tst.label.values, PRED[(src, f1_, a)], PRED[(src, f2_, a)])
            mc.append({'arah': f'{src} → {"D" if src == "C" else "C"}', 'algoritma': a, 'pasangan': f'{FS_NAME[f1_]} vs {FS_NAME[f2_]}',
                       'n01': n01, 'n10': n10, 'p_nilai': round(p, 5), 'signifikan': 'ya' if p < 0.05 else 'tidak'})
mc = pd.DataFrame(mc); save_tab(mc, 'E2_mcnemar', 'E2_mcnemar'); print(mc.to_string(index=False))
# Metrik per kelas untuk RF transfer C → D
pc = []
for fs in ('AE', 'S2s'):
    p_, r_, f_, n_ = precision_recall_fscore_support(Dd.label, PRED[('C', fs, 'RF')], labels=USE, zero_division=0)
    for k, pp, rr, ff, nn in zip(USE, p_, r_, f_, n_):
        pc.append({'set_fitur': FS_NAME[fs], 'kelas': CLASS_ID[k], 'presisi': round(pp, 4), 'recall': round(rr, 4), 'F1': round(ff, 4), 'n': int(nn)})
pc = pd.DataFrame(pc); save_tab(pc, 'E2_metrik_per_kelas_RF_C_ke_D', 'E2_per_class_metrics_RF_C_to_D'); print(pc.to_string(index=False))

## Tahap 7 — E3: Struktur ruang fitur tanpa label
k-means (k = jumlah kelas) pada semua sampel kedua blok, lalu dibandingkan dengan label: **ARI** dan **NMI** (0 = acak, 1 = sempurna). Visualisasi t-SNE 1.500 titik.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.manifold import TSNE
rows = []
for fs, cols in FS.items():
    X = df[cols].values if fs == 'AE' else StandardScaler().fit_transform(df[cols].values)
    for s in range(5):
        km = KMeans(n_clusters=len(USE), n_init=10, random_state=SEED + s).fit(X)
        rows.append({'set_fitur': FS_NAME[fs], 'ulangan': s + 1, 'ARI': adjusted_rand_score(df.label, km.labels_), 'NMI': normalized_mutual_info_score(df.label, km.labels_)})
E3 = pd.DataFrame(rows).groupby('set_fitur', sort=False).agg(ARI=('ARI', 'mean'), ARI_sd=('ARI', 'std'), NMI=('NMI', 'mean'), NMI_sd=('NMI', 'std')).reset_index().round(4)
save_tab(E3, 'E3_klasterisasi', 'E3_clustering'); print(E3.to_string(index=False))
kk = 1500 // (2 * len(USE))
sub = df.loc[np.concatenate([g.sample(min(len(g), kk), random_state=SEED).index.values for _, g in df.groupby(['blok', 'label'])])].reset_index(drop=True)
TS = {}
for fs in ('S2s', 'AE'):
    X = sub[FS[fs]].values if fs == 'AE' else StandardScaler().fit_transform(sub[FS[fs]].values)
    TS[fs] = TSNE(n_components=2, perplexity=30, random_state=SEED, init='pca').fit_transform(X)

## Tahap 8 — Peta klasifikasi Blok C (Random Forest GEE, seluruh kumpulan latih)

In [ ]:
def classify(fs):
    cols = FS[fs]; fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([r.lon, r.lat]), {**{c: float(getattr(r, c)) for c in cols}, 'label': int(r.label)})
                                             for r in POOL[cols + ['lon', 'lat', 'label']].itertuples()])
    clf = ee.Classifier.smileRandomForest(200, seed=SEED).train(fc, 'label', cols)
    return FEAT.select(cols).clip(BLK['C']).classify(clf)
PAL = {1: '1b7837', 2: 'd9c76a', 3: '4fa3c7', 4: 'd6604d', 5: 'f4a6d7'}
pal = [PAL[k] for k in range(1, 6)]
MAPS, AREA = {}, []
for fs in ('S2s', 'AE'):
    img = classify(fs)
    url = img.visualize(min=1, max=5, palette=pal).getThumbURL({'region': BLK['C'], 'dimensions': 900, 'format': 'png', 'crs': CRS})
    fn = os.path.join(DIR_FIG, f'T05_peta_{fs}.png'); open(fn, 'wb').write(requests.get(url, timeout=300).content); MAPS[fs] = fn
    g = ee.Image.pixelArea().divide(1e4).addBands(img).reduceRegion(ee.Reducer.sum().group(1, 'classification'), BLK['C'], 20, maxPixels=1e10, tileScale=8).get('groups').getInfo()
    for x in g: AREA.append({'set_fitur': FS_NAME[fs], 'kelas': CLASS_ID[int(x['classification'])], 'luas_peta_ha': round(x['sum'], 0)})
rgb = s2d.select(['B4_kmr', 'B3_kmr', 'B2_kmr']).visualize(min=0, max=0.15)
open(os.path.join(DIR_FIG, 'T05_s2_blokC.png'), 'wb').write(requests.get(rgb.getThumbURL({'region': BLK['C'], 'dimensions': 900, 'format': 'png', 'crs': CRS}), timeout=300).content)
AREA = pd.DataFrame(AREA); save_tab(AREA, 'luas_peta_blokC', 'map_area_blockC'); print(AREA.pivot(index='kelas', columns='set_fitur', values='luas_peta_ha'))

## Gambar (dua bahasa)

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from matplotlib.patches import Patch
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10})
COLF = {'S2a': '#9e9e9e', 'S2s': '#4C78A8', 'AE': '#E45756', 'AE+S2s': '#54A24B'}
L = {'ID': dict(n='Jumlah titik latih per kelas', f1='F1-makro (Blok C, data uji tetap)', tr='F1-makro', within='Dalam blok', transfer='Transfer ke blok lain',
                cls=CLASS_ID, fs=FS_NAME, s2='Sentinel-2 RGB (kemarau)', mtitle='Peta {}', ari='Skor (k-means vs label)', tsne='t-SNE: {}'),
     'EN': dict(n='Training points per class', f1='Macro-F1 (Block C, fixed test set)', tr='Macro-F1', within='Within block', transfer='Transfer to other block',
                cls=CLASS_EN, fs=FS_EN, s2='Sentinel-2 RGB (dry season)', mtitle='{} map', ari='Score (k-means vs labels)', tsne='t-SNE: {}')}
NPOS = SIZES + [600]
for lang, t in L.items():
    # 1. Kurva efisiensi label
    fig, axs = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
    for ax, a in zip(axs, ALGS):
        for fs in FS:
            s = E1s[(E1s.set_fitur == FS_NAME[fs]) & (E1s.algoritma == a)]
            m, sd = s.F1_makro_rata2.values, s.F1_makro_sd.fillna(0).values
            ax.plot(NPOS, m, 'o-', color=COLF[fs], label=t['fs'][fs], ms=4); ax.fill_between(NPOS, m - sd, m + sd, color=COLF[fs], alpha=0.15)
        ax.set_xscale('log'); ax.set_xticks(NPOS); ax.set_xticklabels([str(x) for x in SIZES] + ['semua' if lang == 'ID' else 'all'], fontsize=8); ax.minorticks_off()
        ax.set_title(a); ax.set_xlabel(t['n']); ax.grid(alpha=0.3)
    axs[0].set_ylabel(t['f1']); axs[-1].legend(fontsize=8, loc='lower right')
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T05_gambar_efisiensi_label_{lang}.png'), dpi=300); plt.close(fig)
    # 2. Transfer
    fig, axs = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
    for ax, ar in zip(axs, ['C → D', 'D → C']):
        s = E2[E2.arah == ar]; x = np.arange(len(ALGS)); w = 0.2
        for i, fs in enumerate(FS):
            v = s[s.set_fitur == FS_NAME[fs]].set_index('algoritma').loc[ALGS]
            ax.bar(x + (i - 1.5) * w, v.F1_transfer, w, color=COLF[fs], label=t['fs'][fs])
            ax.scatter(x + (i - 1.5) * w, v.F1_dalam_blok, marker='_', s=120, color='k', zorder=3)
        ax.set_xticks(x); ax.set_xticklabels(ALGS); ax.set_title(ar); ax.set_ylim(0, 1); ax.grid(axis='y', alpha=0.3)
    axs[0].set_ylabel(t['tr'] + ' (' + t['transfer'] + '; — = ' + t['within'] + ')', fontsize=9); axs[1].legend(fontsize=8, loc='lower right')
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T05_gambar_transfer_{lang}.png'), dpi=300); plt.close(fig)
    # 3. ARI / NMI
    fig, ax = plt.subplots(figsize=(6.5, 3.6)); x = np.arange(len(FS))
    ax.bar(x - 0.18, E3.ARI, 0.36, yerr=E3.ARI_sd, label='ARI', color='#4C78A8'); ax.bar(x + 0.18, E3.NMI, 0.36, yerr=E3.NMI_sd, label='NMI', color='#F58518')
    ax.set_xticks(x); ax.set_xticklabels([t['fs'][k] for k in FS], rotation=12, fontsize=9); ax.set_ylabel(t['ari']); ax.legend(); ax.set_ylim(0, 1)
    fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T05_gambar_klasterisasi_{lang}.png'), dpi=300); plt.close(fig)
    # 4. t-SNE
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.6))
    for ax, fs in zip(axs, ('S2s', 'AE')):
        for k in USE:
            m = sub.label.values == k; ax.scatter(TS[fs][m, 0], TS[fs][m, 1], s=5, color='#' + PAL[k], label=t['cls'][k])
        ax.set_title(t['tsne'].format(t['fs'][fs])); ax.set_xticks([]); ax.set_yticks([])
    axs[1].legend(markerscale=3, fontsize=8, loc='best'); fig.tight_layout(); fig.savefig(os.path.join(DIR_FIG, f'T05_gambar_tsne_{lang}.png'), dpi=300); plt.close(fig)
    # 5. Peta
    fig, axs = plt.subplots(1, 3, figsize=(13, 4.6))
    for ax, (fn, ttl) in zip(axs, [(os.path.join(DIR_FIG, 'T05_s2_blokC.png'), t['s2']), (MAPS['S2s'], t['mtitle'].format(t['fs']['S2s'])), (MAPS['AE'], t['mtitle'].format(t['fs']['AE']))]):
        ax.imshow(Image.open(fn)); ax.set_title(ttl, fontsize=10); ax.axis('off')
    fig.legend(handles=[Patch(color='#' + PAL[k], label=t['cls'][k]) for k in USE], loc='lower center', ncol=len(USE), fontsize=9)
    fig.tight_layout(rect=(0, 0.08, 1, 1)); fig.savefig(os.path.join(DIR_FIG, f'T05_gambar_peta_blokC_{lang}.png'), dpi=250); plt.close(fig)
print(sorted(os.listdir(DIR_FIG)))

In [ ]:
# Ringkasan dan unduh
best = E1s[E1s.n_per_kelas == 'semua'].sort_values('F1_makro_rata2', ascending=False)
summary = {'tahun_embedding': Y, 'tahun_kehilangan_untuk_kelas_dibuka': [2000 + v for v in LOSS_YEARS], 'kelas_dipakai': [CLASS_ID[k] for k in USE],
           'n_sampel_per_blok': df.blok.value_counts().to_dict(), 'n_latih_C': len(POOL), 'n_uji_C': len(TEST),
           'E1_semua_label': best[['set_fitur', 'algoritma', 'F1_makro_rata2']].to_dict('records'),
           'E1_n10': E1s[E1s.n_per_kelas == 10][['set_fitur', 'algoritma', 'F1_makro_rata2', 'F1_makro_sd']].to_dict('records'),
           'E1_label_setara_AE10': eff.to_dict('records'), 'E2_transfer': E2.to_dict('records'), 'E2_mcnemar': mc.to_dict('records'),
           'E3_klasterisasi': E3.to_dict('records'), 'luas_peta_blokC': AREA.to_dict('records'), 'kandidat_blok': blocks.to_dict('records')}
json.dump(summary, open(os.path.join(DIR_TAB, 'T05_hasil_ringkas.json'), 'w'), indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
zpath = '/content/T05_hasil.zip' if os.path.exists('/content') else 'T05_hasil.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs_ in os.walk(OUT):
        for fn in fs_: z.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), os.path.dirname(OUT)))
print('Y =', Y, '| kelas:', summary['kelas_dipakai']); print(best.head(6).to_string(index=False))
try:
    from google.colab import files; files.download(zpath)
except Exception:
    print('Arsip:', zpath)

Selesai. Upload `T05_hasil.zip` ke Claude.